In [0]:
from pyspark.sql.functions import from_json, col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
import uuid

RAW_TABLE = "payment_gateway_catalog.raw.raw_payment_payloads"
BRONZE_TABLE = "payment_gateway_catalog.bronze.bronze_payment_raw"

payload_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("provider", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("data", StructType([
        StructField("payment_id", StringType(), True),
        StructField("transaction_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("account_id", StringType(), True),
        StructField("amount_subunits", LongType(), True),
        StructField("currency", StringType(), True),
        StructField("fx_rate_to_inr", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("method", StringType(), True),
        StructField("vpa", StringType(), True),
        StructField("fee", LongType(), True),
        StructField("tax", LongType(), True),
        StructField("tenure_months", LongType(), True),
        StructField("merchant_category_code", StringType(), True),
        StructField("tax_jurisdiction", StringType(), True),
        StructField("vat_or_sales_tax_collected", LongType(), True),
        StructField("error", StructType([
            StructField("code", StringType(), True),
            StructField("description", StringType(), True)
        ]), True)
    ]), True)
])
batch_id = str(uuid.uuid4())

raw_df = spark.read.table(RAW_TABLE)
parsed_df = raw_df.select(
    from_json(col("raw_payload"), payload_schema).alias("parsed"),
              col("_ingested_at")
).select(
    col("parsed.event_id").alias("event_id"),
    col("parsed.provider").alias("provider"),
    col("parsed.timestamp").alias("timestamp"),
    col("parsed.event_type").alias("event_type"),
    col("parsed.data").alias("data"),
    col("_ingested_at"),
    lit(RAW_TABLE).alias("_source_file"),
    lit(batch_id).alias("_batch_id")
).dropDuplicates(["event_id"])

parsed_df.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("provider") \
    .saveAsTable(BRONZE_TABLE)
print(f"Successfully processed Bronze Layer table: {BRONZE_TABLE}")

In [0]:
df = spark.read.table(BRONZE_TABLE)
display(df.filter(df.provider == "razorpay"))